In [1]:
import torch
import torchvision.transforms as transforms
from torchvision.datasets import Imagenette, SVHN
import numpy as np
import logging
import importlib

import FL_sim
import models_to_train
import components.compression_reporting, components.components

importlib.reload(FL_sim)
importlib.reload(models_to_train)
importlib.reload(components.compression_reporting)
importlib.reload(components.components)

from models_to_train import ResNetPLModel
from FL_sim import FLSimulator, save_grads_f_applied_on_grads
from components.components import wz_encoding_process, wz_reconstruction_process
from components.compression_reporting import report_decompression_stat, report_compression_stat, state_report_per_round

torch.set_float32_matmul_precision('high')

In [2]:
# dataset = [
#     Imagenette(
#         root='./data/Imagenet', split=s,
#         transform=transforms.Compose([
#             transforms.Resize(256),
#             transforms.CenterCrop(224),
#             transforms.ToTensor(),
#             transforms.Normalize(
#                 mean=[0.485, 0.456, 0.406],
#                 std=[0.229, 0.224, 0.225]
#             )
#         ])
#     ) for s in ['train', 'val']]


dataset = [
    SVHN(
        root='./data/SVHN', split=s,
        transform=transforms.Compose([
            transforms.Resize(32),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.4377, 0.4438, 0.4728],
                std=[0.1980, 0.2010, 0.1970]
            ),
        ])
    ) for s in ['train', 'test']]

# dataset = [torch.utils.data.Subset(d, list(range(50))) for d in dataset]
# for i in range(10):
#     for d in dataset:
#         d.dataset.labels[i]=i

In [3]:
importlib.reload(components.compression_reporting)
from components.compression_reporting import state_report_per_round

# @report_compression_stat
def pre_send_process(worker_grad_dict, agent_id):
    # worker_broadcast_data = [worker_grad_dict]
    worker_broadcast_data = wz_encoding_process(worker_grad_dict, agent_id)
    return worker_broadcast_data


# @report_decompression_stat
def server_rec_process(agent_id, worker_count, global_model_dims, previous_data, worker_broadcast_data):
    # result_dict = worker_broadcast_data[0]
    result_dict = wz_reconstruction_process(
        agent_id, worker_count, global_model_dims, previous_data, worker_broadcast_data)
    return result_dict


def applied_on_grads_before_optim(fl_model, worker_id, curr_round, current_epoch, batch_idx, *args, **kwargs):
    # save_grads_f_applied_on_grads(fl_model,
    #     worker_id, curr_round, current_epoch, batch_idx, *args, **kwargs)
    pass

In [4]:
k = 5  # Number of workers

logging_disabled=True
post_training_report=True

k = 3
logging_disabled=False
post_training_report=False

batch_size = 13000
# batch_size /= 6 # more batches as samples
# batch_size /= 50 # imagenet
# batch_size /= 3 # resnet 50
batch_size = int(batch_size)

# model.args_for_f_on_grad['save_folder_path'] = \
#     f"experiments/exp_data/gradients_resnet/gradients_resnet_t{i}/"

model = ResNetPLModel(num_classes=10, resnet_version='resnet18', lr=0.01,
                      logging_disabled=logging_disabled, applied_on_grads_before_optim=applied_on_grads_before_optim)

model.load_state_dict(torch.load('experiments/exp_data/resnet18_svhn.pth', map_location='cpu'))

sim = FLSimulator(
    num_agents=k, communication_rounds=10, client_epochs_per_round=15,
    batch_size=batch_size, dataset_train=dataset[0], dataset_test=dataset[1],
    aggregation_method='fedavg', non_iid_sampling=False, pl_model=model,
    pre_send_process=pre_send_process, server_rec_process=server_rec_process
)

# pre-training global model before saving it
# sim.do_train_global_model_and_set_local_model(num_epochs=4)
# torch.save(sim.global_model.state_dict(), 'experiments/exp_data/resnet18_svhn.pth')

logging.getLogger("pytorch_lightning").setLevel(logging.WARNING)
sim.run_simulation(post_training_report=post_training_report, pre_training_global_epochs=0)


round 1/10 --------------------
  - reporting global model metrics
     > training agent 1/3


Missing logger folder: experiments/exp_data/run_stats\agent_0_round_0
Missing logger folder: experiments/exp_data/run_stats\agent_1_round_0


     > training agent 2/3


Missing logger folder: experiments/exp_data/run_stats\agent_2_round_0


     > training agent 3/3
Aggregating gradients using FedAvg...


...


round 10/10 --------------------
  - reporting global model metrics
     > training agent 1/3


Missing logger folder: experiments/exp_data/run_stats\agent_1_round_9


     > training agent 2/3


Missing logger folder: experiments/exp_data/run_stats\agent_2_round_9


     > training agent 3/3
Aggregating gradients using FedAvg...

final global model metrics


In [5]:
torch.save(
    state_report_per_round,
    f'experiments/exp_data/compression_stats_report/compression_stats.pth',
)